In [1]:
import os 
import glob
from zipfile import ZipFile
import json
working_path = os.getcwd()

In [3]:
def unzip_file(path):
    archive = zipfile.ZipFile(path, 'r')
    result = []
    for fileset in archive.filelist:
        data = archive.read(fileset)
        lines = data.decode().splitlines()
        data = [json.loads(line) for line in lines]
        result.extend(data)
    return result

In [4]:
import glob
import zipfile
import os
path='draft-data/stashed_hourly/'
files = glob.glob(path + '/**/**/**.zip', recursive=True)
data = []
for file in files:
    data.extend(unzip_file(file))

In [5]:
log = []
error = []
for value in data:
    try:
        log.append((value["timestamp_unix"], value["content"]))
    except KeyError:
        error.append(value)

In [31]:
error_perct = 100*len(error)/(len(log)+len(error))

print('Found {}% of error'.format(error_perct))

Found 0.0037398556415722353% of error


In [7]:
log = sorted(log, key=lambda x: float(x[0]))

In [8]:
 one_week = (7 * 24 * 60 * 60)

In [9]:
timestamp,logs = zip(*log)

In [10]:
for i in range(len(timestamp)):
    if timestamp[i]>timestamp[0]+one_week:
        value_index=i
        break

In [11]:
train_log=logs[:value_index]
test_log=logs[value_index:]

In [12]:
os.chdir('offline')
from src.template_extraction.drain import LogParser
from src.template_extraction.mike import Mike

In [13]:
artifacts_path = ""

In [14]:
import pickle
def make_directory(directory):
    if not (os.path.exists(directory)):
        try:
            os.makedirs(directory)
        except:
            pass


def save_object(filepath, fileobject):
    """
    Save python object with pickle
    :param filepath: path for the object
    :type filepath: str
    :param fileobject: Python Object
    :type fileobject: object
    """
    # print(f"Save {os.path.basename(filepath)} at {os.path.dirname(filepath)}")
    with open(filepath, "wb") as f:
        pickle.dump(fileobject, f)


def load_object(filepath):
    """
    Load Pickle Objects
    :param filepath: Path of the file
    :type filepath: str / path
    :return: Object
    :rtype: object
    """
    with open(filepath, "rb") as f:
        obj = pickle.load(f)
    return obj


In [15]:
 def template_extraction(log_messages, depth=4, st=0.4,source="data"):
        parser = LogParser(depth=depth, st=st)
        df_events = parser.parse(log_messages)
        if df_events.empty:
            return False
        else:
            # df_events = Mike(df_events).get_updated_events()
            # Saving
            df_events.to_csv(
                os.path.join(artifacts_path, f"{source}-events.csv"),
                index=False,
            )
            save_object(
                os.path.join(
                    artifacts_path, f"{source}-gale.pickle"
                ),
                parser,
            )
            return True

In [16]:
template_extraction(train_log)

Starting Template Extraction 
0.0M of log lines has been processed
Template Extraction done. [Time taken: 0:00:00.725752]


True

In [17]:
def template_transformer(training_data,source="data"):
        gale = load_object(
            os.path.join(artifacts_path, f"{source}-gale.pickle")
        )
        templates = [gale.predict([logs]) for logs in training_data]
        return templates

In [18]:
templates = template_transformer(test_log)
templates = [t[0] for t in templates]

In [19]:
from collections import Counter
Counter(templates)

Counter({'f5052a98': 4410,
         'bb836e43': 13178,
         'decb4eca': 4408,
         'fbeecffe': 234,
         '6cf35c3c': 35294,
         '88a5587d': 35306,
         '1813df44': 5066,
         '795e5d1a': 150,
         'a52f280b': 52,
         'novelty': 70})

In [32]:
if 'novelty' in Counter(templates).keys():
    print(100*Counter(templates)['novelty']/len(templates))

0.0713063320022818


In [23]:
index = [i for i, e in enumerate(templates) if e == 'novelty']

In [24]:
c = [test_log[i] for i in index]

In [26]:
c

['forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 failed',
 'forecasting for site 765 f